# Care Agents eval

Romario Joseph

Runs the 20 test cases through the baseline (one Gemini 4 Hydrogen call) and the system (Hydrogen intake reader, then Gemini 4 Neon verifier), and prints the results table.

Run the cells in order. Use **Runtime > Run all** if you like.

## 1. Setup

In [ ]:
!pip install -q google-genai python-dotenv requests
!rm -rf gemini4-5sigma-care-agents
!git clone -q https://github.com/romario-joseph/gemini4-5sigma-care-agents
%cd gemini4-5sigma-care-agents
!ls eval

## 2. API key

Add your hackathon key once: click the **key icon** in the left sidebar, add a secret named `GEMINI_API_KEY`, and turn on notebook access. If you skip that, this cell asks for the key instead. The key is never saved in the notebook.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('Key loaded from Colab secrets')
except Exception:
    from getpass import getpass
    os.environ['GEMINI_API_KEY'] = getpass('Paste your GEMINI_API_KEY: ')

## 3. Check the model names

If `gemini-4-hydrogen` or `gemini-4-neon` is not in this list, edit the names at the top of `eval/prompts.py`.

In [ ]:
from google import genai
client = genai.Client()
names = [m.name for m in client.models.list() if 'gemini-4' in m.name]
print('\n'.join(names))

## 4. Photos and voice notes

Copies the files from the **Eval Media** folder in your Google Drive into `eval/media/`. Sign in with the Google account that owns the folder. If the folder is not found, an upload button appears so you can pick the 13 files by hand.

In [ ]:
import glob, shutil
from pathlib import Path
media = Path('eval/media'); media.mkdir(exist_ok=True)
found = []
try:
    from google.colab import drive
    drive.mount('/content/drive')
    hits = glob.glob('/content/drive/MyDrive/**/Eval Media', recursive=True)
    if hits:
        for f in glob.glob(hits[0] + '/C*'):
            shutil.copy(f, media / Path(f).name)
            found.append(Path(f).name)
except Exception as e:
    print('Drive not available:', e)
if not found:
    from google.colab import files
    for name, data in files.upload().items():
        (media / name).write_bytes(data)
        found.append(name)
print(len(found), 'files:', sorted(found))

## 5. Quick check on 3 cases

C01 (messy note), C09 (asks for a dose), C18 (prompt injection). About 2 to 4 minutes.

In [ ]:
!python eval/run_eval.py --cases C01,C09,C18 --no-url-check

## 6. Full run, all 20 cases

About 10 to 20 minutes. Lower `--workers` if you see rate limit errors.

In [ ]:
!python eval/run_eval.py --workers 4

## 7. Results

In [ ]:
import glob, pandas as pd
from IPython.display import Markdown, display
summary = sorted(glob.glob('eval/results/summary_*.md'))[-1]
cases = sorted(glob.glob('eval/results/cases_*.csv'))[-1]
display(Markdown(open(summary).read()))
df = pd.read_csv(cases)
display(df)

## 8. Where it failed

Cases where the system missed needs, got urgency wrong, or failed a safety check. Use these for the "what we fixed" part of the demo.

In [ ]:
sys_rows = df[df.system == 'system']
fails = sys_rows[(sys_rows.need_f1 < 1) | (sys_rows.urgency_acc < 1) | (sys_rows.safety_ok == False)]
display(fails[['case_id','input_used','need_f1','urgency_acc','safety_ok','got_needs','latency_s']])

## 9. Download

Saves the results to your computer. Paste the summary table into the README under Eval.

In [ ]:
from google.colab import files
files.download(summary)
files.download(cases)